In [2]:
import jax
import jax.numpy as jnp
import jax.random as random
from typing import Tuple, Dict, List
from collections import namedtuple
import json
import os

# Constants
SENSORY_TYPES = ("block_type", "position", "state")
ACTION_SPACE = ("move", "mine", "craft")
DRIVE_WEIGHTS = (
    ("hunger", 0.2), ("shelter", 0.3), ("knowledge", 0.2), ("curiosity", 0.2), ("influence", 0.1)
)
RECURSION_DEPTHS = (10, 20, 30, 40, 50)
MEMORY_DIR = "agi_memory/"

# Memory and Reflection Classes
class PersistentMemory:
    def __init__(self):
        self.episodic = []
        self.semantic = {}
        self.identity = {"name": "AGI", "core_objective": "Learn and adapt"}
        os.makedirs(MEMORY_DIR, exist_ok=True)

    def store_episodic(self, event: Dict):
        # Convert JAX arrays to Python types for JSON serialization
        event = {k: int(v) if k == "action_idx" else float(v) if k == "reward" else v for k, v in event.items()}
        self.episodic.append(event)
        with open(f"{MEMORY_DIR}/episodic.json", "w") as f:
            json.dump(self.episodic, f)

    def store_semantic(self, concept: str, value: float):
        self.semantic[concept] = value
        with open(f"{MEMORY_DIR}/semantic.json", "w") as f:
            json.dump(self.semantic, f)

    def query(self, memory_type: str) -> List[Dict]:
        return getattr(self, memory_type, [])

class ReflectionLog:
    def __init__(self):
        self.log = []

    def log_reflection(self, alignment: float, action: str):
        self.log.append({"alignment": alignment, "action": action})

    def evaluate_alignment(self, drives, state, feedback) -> float:
        drive_weights = jnp.array([weight for _, weight in drives])
        alignment = jnp.dot(drive_weights, state[:5]) / (jnp.linalg.norm(drive_weights) + 1e-8)
        return float(alignment)

# GameWorld with O3 Additions (#1, #3, #4, #7)
class GameWorld:
    def __init__(self):
        self.position = jnp.array([0, 0, 0], dtype=jnp.float32)
        self.blocks = jnp.zeros((3, 3, 3))  # Simulated block types
        self.actions_taken = 0
        self.max_actions = 1000
        self.frame_stack = jnp.zeros((4, 9))  # Frame-stacking (#4)
        self.prev_sensory = jnp.zeros(27)  # For curiosity (#3)

    def sense(self) -> jnp.ndarray:
        # Multimodal (#1): Block types, position
        # Proprioception (#7): Position as state signal
        block_data = self.blocks.flatten()[:20]  # (20,)
        position = self.position / jnp.max(jnp.abs(self.position) + 1e-8)  # (3,)
        # Frame-stacking (#4)
        self.frame_stack = jnp.roll(self.frame_stack, -1, axis=0)
        self.frame_stack = self.frame_stack.at[-1].set(block_data[:9])  # Downsample
        sensory_data = jnp.concatenate([self.frame_stack.flatten()[:24], position])  # (27,)
        return sensory_data / jnp.max(sensory_data + 1e-8)

    def act(self, action_idx: int) -> Tuple[float, bool]:
        action = ACTION_SPACE[action_idx]
        reward = 0.1 if action == "move" else 0.2 if action == "mine" else 0.3
        if action == "move":
            self.position += jnp.array([0.1, 0.0, 0.0])
        elif action == "mine":
            self.blocks = self.blocks.at[1, 1, 1].set(1.0)
        self.actions_taken += 1
        success = self.actions_taken < self.max_actions
        return reward, success

    def curiosity_reward(self, state, next_state):
        key = random.PRNGKey(0)
        W = random.normal(key, (27, 27))  # Match sensory shape
        predicted_state = jnp.dot(state, W)  # (27,)
        error = jnp.mean((predicted_state - next_state) ** 2)  # (27,)
        return 0.1 * error  # Curiosity (#3)

    def get_feedback(self) -> Dict:
        next_state = self.sense()  # (27,)
        feedback = {
            "success": self.actions_taken > 0,
            "pain": 0.0 if self.actions_taken < self.max_actions else 1.0,
            "reward": 0.0,  # Placeholder for task reward
            "time_pressure": self.actions_taken / 1000.0
        }
        # Reward shaping (#8)
        task_reward = feedback["reward"]
        curiosity = self.curiosity_reward(self.prev_sensory, next_state)
        alignment = reflection.evaluate_alignment(DRIVE_WEIGHTS, next_state[:5], feedback)
        feedback["reward"] = 0.5 * task_reward + 0.3 * curiosity + 0.2 * alignment
        self.prev_sensory = next_state
        return feedback

# Core AGI Functions with O3 Additions (#2, #5, #6, #8)
def process_sensory_input(sensory_data: jnp.ndarray, sensory_types: Tuple[str, ...]) -> jnp.ndarray:
    return sensory_data

def compute_drive_scores(state: jnp.ndarray, drives: Tuple[Tuple[str, float], ...]) -> jnp.ndarray:
    drive_weights = jnp.array([weight for _, weight in drives])  # (5,)
    scores = state[:len(drives)] * drive_weights  # (5,) * (5,) -> (5,)
    return scores

def recursive_priority_selector(state: jnp.ndarray, depth: int) -> jnp.ndarray:
    scores = compute_drive_scores(state, DRIVE_WEIGHTS)  # (5,)
    for _ in range(depth):
        scores = jnp.tanh(scores + state[:len(DRIVE_WEIGHTS)])  # (5,)
    return scores

def select_action(scores: jnp.ndarray, action_space: Tuple[str, ...], prev_state: jnp.ndarray) -> int:
    # Re-entrant state (#5)
    state_input = prev_state[:5]  # (5,)
    combined_input = jnp.concatenate([scores, state_input])  # (5,) + (5,) -> (10,)
    key = random.PRNGKey(0)
    W = random.normal(key, (10, len(action_space)))  # (10, 3)
    action_scores = jnp.dot(combined_input, W)  # (3,)
    return jnp.argmax(action_scores)

def update_state_with_sensory_memory(batch_input: jnp.ndarray) -> jnp.ndarray:
    # Memory retrieval (#2)
    sensory_data = world.sense()  # (27,)
    recent_memory = memory.query("episodic")[-1] if memory.episodic else {}
    memory_vector = jnp.array([recent_memory.get("reward", 0.0), recent_memory.get("action_idx", 0)])  # (2,)
    state = jnp.concatenate([sensory_data[:52], memory_vector])  # (54,)
    return state

def dppu_with_dynamic_pi_phi(batch_input: jnp.ndarray, depth: int) -> jnp.ndarray:
    state = batch_input
    for _ in range(depth):
        state = recursive_priority_selector(state, depth)
    return state

def update_drive_weights(drives, alignment, reward, learning_rate=0.01):
    # Online plasticity (#6)
    new_drives = [(name, weight + learning_rate * alignment * reward) for name, weight in drives]
    return tuple(new_drives)

# Main AGI Loop
def agi_loop(batch_input: jnp.ndarray, depth: int) -> Tuple[jnp.ndarray, Dict]:
    global DRIVE_WEIGHTS
    state = update_state_with_sensory_memory(batch_input)
    output = dppu_with_dynamic_pi_phi(state, depth)
    scores = compute_drive_scores(state, DRIVE_WEIGHTS)
    action_idx = select_action(scores, ACTION_SPACE, state)
    reward, success = world.act(action_idx)
    # Fix: Convert JAX arrays to Python types for JSON
    event = {"action_idx": int(action_idx), "reward": float(reward), "success": success}
    feedback = world.get_feedback()
    memory.store_episodic(event)
    alignment = reflection.evaluate_alignment(DRIVE_WEIGHTS, state, feedback)
    reflection.log_reflection(alignment, ACTION_SPACE[action_idx])
    DRIVE_WEIGHTS = update_drive_weights(DRIVE_WEIGHTS, alignment, feedback["reward"])  # RL (#6)
    return output, feedback

# Initialize
world = GameWorld()
memory = PersistentMemory()
reflection = ReflectionLog()

# Test Loop
batch_input = jnp.zeros(50000)
for depth in RECURSION_DEPTHS:
    output, feedback = agi_loop(batch_input, depth)
    print(f"Depth: {depth}, Feedback: {feedback}")

Depth: 10, Feedback: {'success': True, 'pain': 0.0, 'reward': Array(0.00111111, dtype=float32), 'time_pressure': 0.001}
Depth: 20, Feedback: {'success': True, 'pain': 0.0, 'reward': Array(0.02722951, dtype=float32), 'time_pressure': 0.002}
Depth: 30, Feedback: {'success': True, 'pain': 0.0, 'reward': Array(0.02722951, dtype=float32), 'time_pressure': 0.003}
Depth: 40, Feedback: {'success': True, 'pain': 0.0, 'reward': Array(0.02722951, dtype=float32), 'time_pressure': 0.004}
Depth: 50, Feedback: {'success': True, 'pain': 0.0, 'reward': Array(0.02722951, dtype=float32), 'time_pressure': 0.005}


In [3]:
import jax
import jax.numpy as jnp
import jax.random as random
from typing import Tuple, Dict, List
from collections import namedtuple
import json
import os
import pickle
import minerl
import gym

# Constants
SENSORY_TYPES = ("pov", "inventory", "compass")
ACTION_SPACE = ("move", "mine", "craft")
DRIVE_WEIGHTS = (
    ("hunger", 0.2), ("shelter", 0.3), ("knowledge", 0.2), ("curiosity", 0.2), ("influence", 0.1)
)
RECURSION_DEPTHS = (10, 20, 30, 40, 50)
MEMORY_DIR = "agi_memory/"
PARAMS_PATH = f"{MEMORY_DIR}/params.pkl"
global_key = random.PRNGKey(42)  # Fix 1: Global PRNG key

# Memory and Reflection Classes
class PersistentMemory:
    def __init__(self):
        self.episodic = []
        self.semantic = {}
        self.identity = {"name": "AGI", "core_objective": "Learn and adapt"}
        os.makedirs(MEMORY_DIR, exist_ok=True)

    def store_episodic(self, event: Dict):
        event = {k: int(v) if k == "action_idx" else float(v) if k == "reward" else v for k, v in event.items()}
        self.episodic.append(event)
        with open(f"{MEMORY_DIR}/episodic.json", "w") as f:
            json.dump(self.episodic, f)

    def store_semantic(self, concept: str, value: float):
        self.semantic[concept] = value
        with open(f"{MEMORY_DIR}/semantic.json", "w") as f:
            json.dump(self.semantic, f)

    def query(self, memory_type: str) -> List[Dict]:
        return getattr(self, memory_type, [])

class ReflectionLog:
    def __init__(self):
        self.log = []

    def log_reflection(self, alignment: float, action: str):
        self.log.append({"alignment": alignment, "action": action})

    def evaluate_alignment(self, drives, state, feedback) -> float:
        drive_weights = jnp.array([weight for _, weight in drives])
        alignment = jnp.dot(drive_weights, state[:5]) / (jnp.linalg.norm(drive_weights) + 1e-8)
        return float(alignment)

# MineRL GameWorld with O3 Additions (#1, #3, #4, #7)
class GameWorld:
    def __init__(self, env_name: str = "MineRLNavigateDense-v0"):
        self.env = gym.make(env_name)
        self.state = self.env.reset()
        self.actions_taken = 0
        self.max_actions = 1000
        self.frame_stack = jnp.zeros((4, 16))  # Frame-stacking (#4)
        self.prev_sensory = jnp.zeros(27)  # For curiosity (#3)
        # Fix 3: Learnable curiosity model
        self.curiosity_params = {"W": random.normal(random.PRNGKey(42), (27, 27))}
        # Fix 5: Load parameters
        if os.path.exists(PARAMS_PATH):
            with open(PARAMS_PATH, "rb") as f:
                params = pickle.load(f)
                self.curiosity_params = params.get("curiosity_params", self.curiosity_params)

    def sense(self) -> jnp.ndarray:
        # Multimodal (#1, #7): POV, inventory, compass
        pov = self.state["pov"].mean(axis=-1)[:4, :4].flatten()  # Downsample to (16,)
        self.frame_stack = jnp.roll(self.frame_stack, -1, axis=0)
        self.frame_stack = self.frame_stack.at[-1].set(pov)
        inventory = jnp.array([self.state["inventory"].get(k, 0) for k in ["wood", "plank", "stick"]])  # (3,)
        compass = jnp.array([self.state["compass"]["angle"]])  # (1,)
        sensory_data = jnp.concatenate([self.frame_stack.flatten()[:23], inventory, compass])  # (27,)
        return sensory_data / jnp.max(sensory_data + 1e-8)

    def act(self, action_idx: int) -> Tuple[float, bool]:
        action_map = {
            0: {"forward": 1, "jump": 0, "attack": 0},  # Move
            1: {"forward": 0, "jump": 0, "attack": 1},  # Mine
            2: {"forward": 0, "jump": 0, "craft": "plank"}  # Craft
        }
        action = action_map[action_idx]
        self.state, reward, done, _ = self.env.step(action)
        self.actions_taken += 1
        return float(reward), not done

    def curiosity_reward(self, state, next_state):
        global global_key
        # Fix 3: Use learnable W, compute loss
        W = self.curiosity_params["W"]
        predicted_state = jnp.dot(state, W)  # (27,)
        loss = jnp.mean((predicted_state - next_state) ** 2)
        # Fix 1: Fresh subkey for noise
        global_key, sub = random.split(global_key)
        noise = random.normal(sub, (27,)) * 0.01
        return 0.1 * loss + noise  # Curiosity (#3)

    def update_curiosity(self, state, next_state, step):
        # Fix 3: SGD update every K=10 steps
        if step % 10 == 0:
            W = self.curiosity_params["W"]
            predicted_state = jnp.dot(state, W)
            loss = jnp.mean((predicted_state - next_state) ** 2)
            grad_W = jax.grad(lambda w: jnp.mean((jnp.dot(state, w) - next_state) ** 2))(W)
            self.curiosity_params["W"] = W - 0.01 * grad_W  # SGD step

    def get_feedback(self) -> Dict:
        next_state = self.sense()  # (27,)
        feedback = {
            "success": self.actions_taken > 0,
            "pain": 0.0 if self.actions_taken < self.max_actions else 1.0,
            "reward": float(self.state.get("reward", 0.0)),
            "time_pressure": self.actions_taken / 1000.0,
            # Stub for dialog head: Simple inventory summary
            "dialog": f"Inventory: wood={self.state['inventory'].get('wood', 0)}"
        }
        # Reward shaping (#8)
        task_reward = feedback["reward"]
        curiosity = self.curiosity_reward(self.prev_sensory, next_state)
        alignment = reflection.evaluate_alignment(DRIVE_WEIGHTS, next_state[:5], feedback)
        feedback["reward"] = 0.5 * task_reward + 0.3 * curiosity + 0.2 * alignment
        self.prev_sensory = next_state
        return feedback

# Core AGI Functions with O3 Additions (#2, #5, #6, #8)
def process_sensory_input(sensory_data: jnp.ndarray, sensory_types: Tuple[str, ...]) -> jnp.ndarray:
    return sensory_data

def compute_drive_scores(state: jnp.ndarray, drives: Tuple[Tuple[str, float], ...]) -> jnp.ndarray:
    drive_weights = jnp.array([weight for _, weight in drives])  # (5,)
    scores = state[:len(drives)] * drive_weights  # (5,) * (5,) -> (5,)
    return scores

def recursive_priority_selector(state: jnp.ndarray, depth: int) -> jnp.ndarray:
    scores = compute_drive_scores(state, DRIVE_WEIGHTS)  # (5,)
    for _ in range(depth):
        scores = jnp.tanh(scores + state[:len(DRIVE_WEIGHTS)])  # (5,)
    return scores

def select_action(scores: jnp.ndarray, action_space: Tuple[str, ...], prev_state: jnp.ndarray) -> int:
    # Re-entrant state (#5), REINFORCE stub
    global global_key
    state_input = prev_state[:5]  # (5,)
    combined_input = jnp.concatenate([scores, state_input])  # (5,) + (5,) -> (10,)
    global_key, sub = random.split(global_key)  # Fix 1: Fresh subkey
    W = random.normal(sub, (10, len(action_space)))  # (10, 3)
    logits = jnp.dot(combined_input, W)  # (3,)
    probs = jnp.softmax(logits)
    action_idx = random.choice(sub, len(action_space), p=probs)
    return int(action_idx)  # REINFORCE-ready

def update_state_with_sensory_memory(batch_input: jnp.ndarray) -> jnp.ndarray:
    # Memory retrieval (#2)
    sensory_data = world.sense()  # (27,)
    recent_memory = memory.query("episodic")[-1] if memory.episodic else {}
    memory_vector = jnp.array([recent_memory.get("reward", 0.0), recent_memory.get("action_idx", 0)])  # (2,)
    state = jnp.concatenate([sensory_data, memory_vector])  # Fix 2: (29,)
    return state

def dppu_with_dynamic_pi_phi(batch_input: jnp.ndarray, depth: int) -> jnp.ndarray:
    state = batch_input
    for _ in range(depth):
        state = recursive_priority_selector(state, depth)
    return state

def update_drive_weights(drives, alignment, reward, learning_rate=0.01):
    # Online plasticity (#6)
    new_drives = [
        (name, jnp.clip(weight + learning_rate * alignment * reward, 0.05, 1.0))  # Fix 4: Clip
        for name, weight in drives
    ]
    return tuple(new_drives)

# Main AGI Loop
def agi_loop(batch_input: jnp.ndarray, depth: int, step: int) -> Tuple[jnp.ndarray, Dict]:
    global DRIVE_WEIGHTS
    state = update_state_with_sensory_memory(batch_input)
    output = dppu_with_dynamic_pi_phi(state, depth)
    scores = compute_drive_scores(state, DRIVE_WEIGHTS)
    action_idx = select_action(scores, ACTION_SPACE, state)
    reward, success = world.act(action_idx)
    event = {"action_idx": int(action_idx), "reward": float(reward), "success": success}
    feedback = world.get_feedback()
    memory.store_episodic(event)
    # Fix 3: Update curiosity model
    world.update_curiosity(world.prev_sensory, feedback["next_state"], step)
    alignment = reflection.evaluate_alignment(DRIVE_WEIGHTS, state, feedback)
    reflection.log_reflection(alignment, ACTION_SPACE[action_idx])
    DRIVE_WEIGHTS = update_drive_weights(DRIVE_WEIGHTS, alignment, feedback["reward"])
    # Fix 5: Save parameters every N=10 steps
    if step % 10 == 0:
        params = {"DRIVE_WEIGHTS": DRIVE_WEIGHTS, "curiosity_params": world.curiosity_params}
        with open(PARAMS_PATH, "wb") as f:
            pickle.dump(params, f)
    return output, feedback

# Initialize
world = GameWorld()
memory = PersistentMemory()
reflection = ReflectionLog()
# Fix 5: Load DRIVE_WEIGHTS
if os.path.exists(PARAMS_PATH):
    with open(PARAMS_PATH, "rb") as f:
        params = pickle.load(f)
        DRIVE_WEIGHTS = params.get("DRIVE_WEIGHTS", DRIVE_WEIGHTS)

# Test Loop
batch_input = jnp.zeros(50000)
for step, depth in enumerate(RECURSION_DEPTHS):
    output, feedback = agi_loop(batch_input, depth, step)
    print(f"Depth: {depth}, Feedback: {feedback}")

ModuleNotFoundError: No module named 'minerl'